# 💻 Laptop Spec Tier Classifier — Exploratory Data Analysis

This notebook explores the raw `laptops.csv` dataset before any feature engineering.
We look at distributions, missing values, correlations and build intuition for the feature engineering choices made in `app.py`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 60)

## 1. Load Dataset

In [ ]:
# Load from data/ folder (run from project root)
df = pd.read_csv('../data/laptops.csv')
print(f'Shape: {df.shape}')
df.head()

## 2. Dataset Overview

In [ ]:
print('Columns:', df.columns.tolist())
print('\nData Types:')
print(df.dtypes)
print('\nNull Values:')
print(df.isnull().sum())

In [ ]:
df.describe(include='all')

## 3. Brand Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df['CompanyName'].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_title('Laptop Count by Brand', fontsize=14, fontweight='bold')
ax.set_xlabel('Brand')
ax.set_ylabel('Count')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 4. Laptop Type Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
df['TypeOfLaptop'].value_counts().plot(kind='barh', ax=ax, color='coral', edgecolor='black')
ax.set_title('Distribution by Laptop Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Count')
plt.tight_layout()
plt.show()

## 5. RAM Distribution (raw)

In [ ]:
# Extract numeric RAM
df['Ram_GB'] = df['Ram'].str.extract(r'(\d+)').astype(int)

fig, ax = plt.subplots(figsize=(8, 4))
df['Ram_GB'].value_counts().sort_index().plot(kind='bar', ax=ax, color='mediumseagreen', edgecolor='black')
ax.set_title('RAM Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('RAM (GB)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

print(df['Ram_GB'].describe())

## 6. Operating System Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
df['OpSys'].value_counts().plot(kind='pie', ax=ax, autopct='%1.1f%%', startangle=90)
ax.set_ylabel('')
ax.set_title('OS Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Screen Size Distribution

In [ ]:
df['Inches_num'] = pd.to_numeric(df['Inches'], errors='coerce')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['Inches_num'].dropna(), bins=20, color='orchid', edgecolor='black')
ax.set_title('Screen Size Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Screen Size (inches)')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 8. Storage Patterns (raw strings)

In [ ]:
# What storage types are most common?
storage_type_flags = {
    'NVMe / PCIe': df['Memory'].str.upper().str.contains('NVME|PCIE').sum(),
    'SSD':         df['Memory'].str.upper().str.contains('SSD').sum(),
    'HDD':         df['Memory'].str.upper().str.contains('HDD').sum(),
    'eMMC':        df['Memory'].str.upper().str.contains('EMMC').sum(),
    'Fusion Drive':df['Memory'].str.upper().str.contains('FUSION').sum(),
    'Flash':       df['Memory'].str.upper().str.contains('FLASH').sum(),
}

pd.Series(storage_type_flags).sort_values(ascending=False).plot(
    kind='bar', figsize=(8,4), color='teal', edgecolor='black', title='Storage Types in Dataset'
)
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 9. CPU Tier Analysis

In [ ]:
def cpu_tier(c):
    c = str(c).upper()
    if 'XEON' in c or 'I7' in c: return 'i7 / Xeon (Tier 3)'
    elif 'I5' in c: return 'i5 (Tier 2)'
    elif 'I3' in c: return 'i3 (Tier 1)'
    return 'Entry / Atom / Celeron (Tier 0)'

df['CPU_Tier_Label'] = df['Cpu'].apply(cpu_tier)
df['CPU_Tier_Label'].value_counts().plot(
    kind='bar', figsize=(8,4), color='royalblue', edgecolor='black',
    title='CPU Tier Distribution'
)
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

## 10. GPU Distribution

In [ ]:
gpu_brand = df['Gpu'].apply(
    lambda x: 'Nvidia' if 'NVIDIA' in str(x).upper()
    else ('AMD' if 'AMD' in str(x) else 'Intel')
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
gpu_brand.value_counts().plot(kind='pie', ax=axes[0], autopct='%1.1f%%', title='GPU Brand Split')
axes[0].set_ylabel('')

gpu_tier_map = df['Gpu'].apply(
    lambda g: 'High (RX 5600M)' if 'RX 5600' in str(g)
    else ('Mid (GTX 1650)' if 'GTX 1650' in str(g) else 'Integrated')
)
gpu_tier_map.value_counts().plot(kind='bar', ax=axes[1], color=['#3B82F6','#F59E0B','#10B981'],
                                  edgecolor='black', title='GPU Tier Distribution')
axes[1].set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. SpecScore and Tier Labels

In [ ]:
# Re-run a quick version of feature engineering to get SpecScore
import sys, os
sys.path.insert(0, os.path.abspath('..'))
from train import engineer_features

df_eng = engineer_features(pd.read_csv('../data/laptops.csv'))
df_eng['SpecTier'] = pd.qcut(
    df_eng['SpecScore'], q=3,
    labels=['Basic', 'Mainstream', 'Powerhouse'],
    duplicates='drop'
)

print('Tier Distribution:')
print(df_eng['SpecTier'].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_eng['SpecTier'].value_counts().plot(kind='bar', ax=axes[0],
    color=['#3B82F6','#F59E0B','#10B981'], edgecolor='black', title='Tier Label Distribution')
axes[0].set_ylabel('Count')
plt.sca(axes[0]); plt.xticks(rotation=0)

axes[1].hist(df_eng['SpecScore'], bins=40, color='mediumpurple', edgecolor='black')
axes[1].set_title('SpecScore Distribution')
axes[1].set_xlabel('SpecScore')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 12. Feature Correlations

In [ ]:
numeric_cols = ['Ram_GB', 'Storage_GB', 'StorageSpeed', 'CPU_Tier', 'GPU_Tier',
                'Is_Touchscreen', 'Is_IPS', 'Is_4K', 'Is_2K', 'Is_Retina',
                'Inches', 'Weight_kg', 'PowerScore', 'SpecScore']

corr = df_eng[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=ax,
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Heatmap', fontsize=14, fontweight='bold', pad=14)
plt.tight_layout()
plt.show()

## Summary

Key takeaways from EDA:
- The dataset is skewed toward certain brands (Dell, Lenovo dominate)
- Most laptops have 8GB RAM; 16GB is the next most common
- SSD is the most common storage type; NVMe is growing
- Intel i5 is the most common CPU tier
- ~80% of GPUs are integrated; discrete GPUs are minority
- SpecScore has a right-skewed distribution; qcut ensures balanced tiers
- PowerScore and SpecScore are highly correlated (expected – this is the label leakage risk)